# Preparation

In [1]:
TESTS = [
    {
        "id": "q1",
        "question": "把 1 到 1000 中，既能被 3 整除又不能被 5 整除的整数个数算出来。只输出最终数字。",
        "gold": "267",
    },
    {
        "id": "q2",
        "question": "某商品先涨价 20%，再打 8 折，最后再涨价 25%。最终价格与原价相比是涨了、跌了还是不变？只回答“涨了X% / 跌了X% / 不变”。",
        "gold": "涨了20%",
    },
    {
        "id": "q3",
        "question": "A 比 B 大，B 比 C 大，C 不比 D 大。以下哪项一定为真？1. A 比 D 大 2. D 比 A 小 3. B 比 D 大 4. 无法确定。只输出选项编号。",
        "gold": "4",
    },
    {
        "id": "q4",
        "question": "5 个人排成一排，甲乙必须相邻，丙不能站两端。共有多少种排法？只输出数字。",
        "gold": "24",
    },
    {
        "id": "q5",
        "question": "有红、蓝、绿三种球各 4 个，从中任取 5 个。至少有 2 个同色是否必然成立？请只回答“是”或“否”，并给一句理由。",
        "gold": "是",
    },
    {
        "id": "q6",
        "question": "一个会议安排在周一到周五。已知：不在周一；如果在周三，则小王不能参加；小王必须参加；周五会议室关闭；周二主讲人出差。问会议只能安排在哪一天？只输出星期几。",
        "gold": "周四",
    },
    {
        "id": "q7",
        "question": """阅读下面 Python 代码，输出什么，只写最终输出内容：

def f(x, acc=[]):
    acc.append(x)
    return acc

print(f(1))
print(f(2))
print(f(3, []))
print(f(4))
""",
        "gold": "[1]\n[1, 2]\n[3]\n[1, 2, 4]",
    },
    {
        "id": "q8",
        "question": """下面代码的时间复杂度是多少？只回答 O(...)。

count = 0
for i in range(n):
    j = 1
    while j < n:
        count += 1
        j *= 2
""",
        "gold": "O(n log n)",
    },
    {
        "id": "q9",
        "question": """下面函数想返回列表中第二大的不同元素，但有 bug。请指出 bug 本质，只用一句话回答。

def second_largest(nums):
    nums.sort()
    return nums[-2]
""",
        "gold": "没有处理重复值",
    },
    {
        "id": "q10",
        "question": "一辆车从 A 到 B，去程速度 60，回程速度 40。求往返平均速度。只输出数字，不要解释。",
        "gold": "48",
    },
    {
        "id": "q11",
        "question": "如果今天是星期二，那么从今天起第 100 天是星期几？只输出星期几。",
        "gold": "星期四",
    },
    {
        "id": "q12",
        "question": "有 8 升和 5 升两个空壶，以及无限水源。请问是否能精确量出 1 升？只回答“能/不能”，再给最短步骤。",
        "gold": "能",
    },
]

In [2]:
def build_prompt(question: str) -> str:
    return f"""你将回答一道推理题。

要求：
1. 可以思考，但不要长篇分析。
2. 如果得出答案，请立刻结束。
3. 最后一行必须输出一个 JSON 对象。
4. JSON 格式必须是：{{"final_answer":"你的答案"}}
5. 不要在最后一行输出其他内容。

题目如下：
{question}
"""

In [3]:
import json
import re
import time
from pathlib import Path

import pandas as pd
from mlx_lm import load, generate

In [4]:
def extract_final_answer_json(text: str) -> str:
    lines = [x.strip() for x in text.splitlines() if x.strip()]
    for line in reversed(lines):
        try:
            obj = json.loads(line)
            if isinstance(obj, dict) and "final_answer" in obj:
                return str(obj["final_answer"]).strip()
        except Exception:
            pass

    m = re.findall(r'(?im)^\s*FINAL\s*:\s*(.+?)\s*$', text)
    if m:
        return m[-1].strip()

    return lines[-1] if lines else ""

In [5]:
def run_one_model(model_name: str, tests, max_tokens: int = 512):
    print(f"Loading model: {model_name}")
    model, tokenizer = load(model_name)
    print("Model loaded.\n")

    results = []

    for i, item in enumerate(tests, 1):
        prompt_text = build_prompt(item["question"])
        messages = [{"role": "user", "content": prompt_text}]
        prompt = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            return_dict=False,
            enable_thinking=False,
        )

        print(f"[{i}/{len(tests)}] Running {item['id']} ...")
        t0 = time.time()
        out = generate(
            model,
            tokenizer,
            prompt=prompt,
            max_tokens=max_tokens,
            verbose=False,
        )
        latency = time.time() - t0

        text = out.text if hasattr(out, "text") else str(out)
        text = text.strip()
        final_pred = extract_final_answer_json(text)

        results.append({
            "model": model_name,
            "qid": item["id"],
            "question": item["question"],
            "gold": item["gold"],
            "pred": text,
            "final_pred": final_pred,
            "latency_sec": round(latency, 2),
            "pred_len": len(text),
            "final_pred_len": len(final_pred),
        })

    return results

# Model Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit

In [ ]:
MODEL_A = "mlx-community/Qwen3.5-27B-Claude-4.6-Opus-Distilled-MLX-4bit"

results_a = run_one_model(
    model_name=MODEL_A,
    tests=TESTS,
    max_tokens=2048,
)

In [ ]:
Path("benchmark_results").mkdir(exist_ok=True)

with open("benchmark_results/results_model_a.json", "w", encoding="utf-8") as f:
    json.dump(results_a, f, ensure_ascii=False, indent=2)

pd.DataFrame(results_a)

# Model Qwen3.5-35B-A3B-4bit

In [6]:
MODEL_B = "mlx-community/Qwen3.5-35B-A3B-4bit"

results_b = run_one_model(
    model_name=MODEL_B,
    tests=TESTS,
    max_tokens=256,
)

Loading model: mlx-community/Qwen3.5-35B-A3B-4bit


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Model loaded.

[1/12] Running q1 ...
[2/12] Running q2 ...
[3/12] Running q3 ...
[4/12] Running q4 ...
[5/12] Running q5 ...
[6/12] Running q6 ...
[7/12] Running q7 ...
[8/12] Running q8 ...
[9/12] Running q9 ...
[10/12] Running q10 ...
[11/12] Running q11 ...
[12/12] Running q12 ...


In [7]:
with open("benchmark_results/results_model_b.json", "w", encoding="utf-8") as f:
    json.dump(results_b, f, ensure_ascii=False, indent=2)

pd.DataFrame(results_b)

,model,qid,question,gold,pred,final_pred,latency_sec,pred_len,final_pred_len
0,mlx-community/Qwen3.5-35B-A3B-4bit,q1,把 1 到 1000 中，既能被 3 整除又不能被 5 整除的整数个数算出来。只输出最终数字。,267,1. 计算 1 到 1000 中能被 3 整除的数：$\lfloor 1000/3 \rfl...,267,2.20,227,3
1,mlx-community/Qwen3.5-35B-A3B-4bit,q2,某商品先涨价 20%，再打 8 折，最后再涨价 25%。最终价格与原价相比是涨了、跌了还是不...,涨了20%,设原价为 1。\n1. 先涨价 20%：$1 \times (1 + 20\%) = 1.2...,涨了20%,1.42,202,5
2,mlx-community/Qwen3.5-35B-A3B-4bit,q3,A 比 B 大，B 比 C 大，C 不比 D 大。以下哪项一定为真？1. A 比 D 大 2...,4,"{""final_answer"":""4""}",4,0.32,20,1
3,mlx-community/Qwen3.5-35B-A3B-4bit,q4,5 个人排成一排，甲乙必须相邻，丙不能站两端。共有多少种排法？只输出数字。,24,18,18,0.27,2,2
4,mlx-community/Qwen3.5-35B-A3B-4bit,q5,有红、蓝、绿三种球各 4 个，从中任取 5 个。至少有 2 个同色是否必然成立？请只回答“是...,是,是，因为最坏情况是取到3种不同颜色的球各1个，第4个球必然与其中一种颜色重复，而取5个球时必...,是,0.66,78,1
5,mlx-community/Qwen3.5-35B-A3B-4bit,q6,一个会议安排在周一到周五。已知：不在周一；如果在周三，则小王不能参加；小王必须参加；周五会议...,周四,"会议只能安排在周四。\n\n{""final_answer"":""周四""}",周四,0.36,33,2
6,mlx-community/Qwen3.5-35B-A3B-4bit,q7,"阅读下面 Python 代码，输出什么，只写最终输出内容：\n\ndef f(x, acc=...","[1]\n[1, 2]\n[3]\n[1, 2, 4]","[1]\n[1, 2]\n[4]\n[1, 2, 4]\n\n{""final_answer""...","[1]\n[1, 2]\n[4]\n[1, 2, 4]",0.69,75,27
7,mlx-community/Qwen3.5-35B-A3B-4bit,q8,下面代码的时间复杂度是多少？只回答 O(...)。\n\ncount = 0\nfor i ...,O(n log n),"O(n log n)\n{""final_answer"":""O(n log n)""}",O(n log n),0.39,40,10
8,mlx-community/Qwen3.5-35B-A3B-4bit,q9,下面函数想返回列表中第二大的不同元素，但有 bug。请指出 bug 本质，只用一句话回答。\...,没有处理重复值,"该函数未处理列表中存在重复元素导致第二大不同元素缺失的情况。\n\n{""final_answ...",未处理重复元素导致第二大不同元素缺失,0.50,69,18
9,mlx-community/Qwen3.5-35B-A3B-4bit,q10,一辆车从 A 到 B，去程速度 60，回程速度 40。求往返平均速度。只输出数字，不要解释。,48,"48\n{""final_answer"":""48""}",48,0.34,24,2


# Comparison

In [8]:
with open("benchmark_results/results_model_a.json", "r", encoding="utf-8") as f:
    results_a = json.load(f)

with open("benchmark_results/results_model_b.json", "r", encoding="utf-8") as f:
    results_b = json.load(f)

df_a = pd.DataFrame(results_a)
df_b = pd.DataFrame(results_b)

df_a[["qid", "gold", "final_pred", "latency_sec", "final_pred_len"]]

,qid,gold,final_pred,latency_sec,final_pred_len
0,q1,267,267,25.29,3
1,q2,涨了20%,涨了20%,32.14,5
2,q3,4,4,20.67,1
3,q4,24,24,40.29,2
4,q5,是,是,13.97,1
5,q6,周四,周四,15.19,2
6,q7,"[1]\n[1, 2]\n[3]\n[1, 2, 4]","[1]\n[1, 2]\n[3]\n[1, 2, 4]",17.08,24
7,q8,O(n log n),O(n log n),8.21,10
8,q9,没有处理重复值,该函数没有先去重，当列表中存在重复的最大值时，会错误地返回最大值本身而非第二大的不同元素。,18.91,45
9,q10,48,48,7.92,2


In [9]:
def normalize_text(s: str) -> str:
    s = s.strip()
    s = s.replace("（", "(").replace("）", ")")
    s = s.replace("：", ":")
    s = re.sub(r"\s+", " ", s)
    return s

def score_row(qid: str, gold: str, pred: str) -> int:
    g = normalize_text(gold)
    p = normalize_text(pred)

    # 精确类题目
    exact_match_ids = {"q1", "q2", "q3", "q4", "q6", "q8", "q10", "q11"}
    if qid in exact_match_ids:
        return int(g == p)

    # 包含关键短语即可
    if qid == "q5":
        return int("是" in p)

    if qid == "q9":
        keys = ["重复", "第二大不同", "没有处理重复值"]
        return int(any(k in p for k in keys))

    if qid == "q12":
        return int("能" in p)

    if qid == "q7":
        # 对代码输出题做宽松匹配
        want_parts = ["[1]", "[1, 2]", "[3]", "[1, 2, 4]"]
        return int(all(x in pred for x in want_parts))

    return 0

df_a["correct"] = df_a.apply(lambda r: score_row(r["qid"], r["gold"], r["final_pred"]), axis=1)
df_b["correct"] = df_b.apply(lambda r: score_row(r["qid"], r["gold"], r["final_pred"]), axis=1)

In [10]:
summary = pd.DataFrame([
    {
        "model": df_a["model"].iloc[0],
        "correct_count": int(df_a["correct"].sum()),
        "avg_latency_sec": round(df_a["latency_sec"].mean(), 2),
        "avg_final_pred_len": round(df_a["final_pred_len"].mean(), 1),
    },
    {
        "model": df_b["model"].iloc[0],
        "correct_count": int(df_b["correct"].sum()),
        "avg_latency_sec": round(df_b["latency_sec"].mean(), 2),
        "avg_final_pred_len": round(df_b["final_pred_len"].mean(), 1),
    }
])

summary

,model,correct_count,avg_latency_sec,avg_final_pred_len
0,mlx-community/Qwen3.5-27B-Claude-4.6-Opus-Dist...,12,20.18,8.2
1,mlx-community/Qwen3.5-35B-A3B-4bit,10,0.71,6.2


In [11]:
compare = df_a[["qid", "gold", "final_pred", "correct", "latency_sec"]].rename(columns={
    "final_pred": "final_pred_a",
    "correct": "correct_a",
    "latency_sec": "latency_a",
}).merge(
    df_b[["qid", "final_pred", "correct", "latency_sec"]].rename(columns={
        "final_pred": "final_pred_b",
        "correct": "correct_b",
        "latency_sec": "latency_b",
    }),
    on="qid",
)

compare

,qid,gold,final_pred_a,correct_a,latency_a,final_pred_b,correct_b,latency_b
0,q1,267,267,1,25.29,267,1,2.20
1,q2,涨了20%,涨了20%,1,32.14,涨了20%,1,1.42
2,q3,4,4,1,20.67,4,1,0.32
3,q4,24,24,1,40.29,18,0,0.27
4,q5,是,是,1,13.97,是,1,0.66
5,q6,周四,周四,1,15.19,周四,1,0.36
6,q7,"[1]\n[1, 2]\n[3]\n[1, 2, 4]","[1]\n[1, 2]\n[3]\n[1, 2, 4]",1,17.08,"[1]\n[1, 2]\n[4]\n[1, 2, 4]",0,0.69
7,q8,O(n log n),O(n log n),1,8.21,O(n log n),1,0.39
8,q9,没有处理重复值,该函数没有先去重，当列表中存在重复的最大值时，会错误地返回最大值本身而非第二大的不同元素。,1,18.91,未处理重复元素导致第二大不同元素缺失,1,0.50
9,q10,48,48,1,7.92,48,1,0.34
